# 2. Data curation: Tukey loop-length filters <a id="2"></a>
In this section we will curate our kinase dataset for input into conformational analysis.


## Table of contents

- [2.1 Kinase taxonomy](#13)
- [2.2 Filtering for activation loop](#22)
  - [2.3.1 Gap-length filter](#231gap)
- [2.3 Tukey activation-loop length filter](#23bounds)
- [2.4 Tukey k-factor scan and eyeball comparison](#24tukey)
  - [2.4.0 Setup: paths and k grid](#2-4-0-setup-paths-and-k-grid)
  - [2.4.1 Load loop-length distribution](#2-4-1-load-loop-length-distribution)
  - [2.4.2 k-factor parameter scan](#2-4-2-k-factor-parameter-scan)
  - [2.4.3 Load pre-computed scan results](#2-4-3-load-pre-computed-scan-results)
  - [2.4.4 Analyse scan](#2-4-4-analyse-scan)
  - [2.4.5 Eyeball (18–32) vs production Tukey (`k=0.8`)](#2-4-5-eyeball-18-32-vs-production-tukey-k-0-8)


## Backend map

How this notebook connects to `workflow/` modules:

![Backend map](images/backend_maps/03b-TukeyLoopLengthFilters.svg)

<!-- mermaid source (GitHub does not render mermaid in .ipynb; SVG above is for GitHub):
```mermaid
flowchart LR
  nb["03b-TukeyLoopLengthFilters"]
  m0["workflow.ca_stripper"]
  nb --> m0
  m1["workflow.kinaseGroupLabelling"]
  nb --> m1
  m2["workflow.reconstruct"]
  nb --> m2
  m3["workflow.utilities"]
  nb --> m3
```
-->


Our data curation pipeline is subdivided in the following sections:
2. [Data curation](#2)
    2.2. [Filtering for activation loop](#22)
    2.3. [Tukey activation-loop length filter](#23bounds)
    2.3.1. [Gap-length filter](#231gap)
    2.4. [Tukey k-factor scan and eyeball comparison](#24tukey)

To get started, let's load some packages!

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

from workflow.ca_stripper import OutlierStripper
from workflow.kinaseGroupLabelling import KinaseGroupLabeller
from workflow.reconstruct import ProteinReconstructor
from workflow.utilities import PDBDownloader
from workflow.utilities import count_pdb_files, braf_res, clear_and_make, make_seg, copy_filtered_pdbs, copy_motif_filtered_datasets


## 2.1 Kinase taxonomy <a id="13"></a>
Annotate each extracted chain in `Results/InterPro_protein_chains/` (`PDB_CHAIN.pdb`) with UniProt / KinHub Manning metadata and UniProt CAUTION pseudokinase flags. Writes `Results/kinase_annotation_all_chains.csv` and `Results/excluded_pseudokinase_basenames.txt`, then shows kinome-group and species %-bar charts plus a pseudokinase pie.


In [ ]:
from workflow.kinaseGroupLabelling import KinaseGroupLabeller

lab = KinaseGroupLabeller()
annot_all = lab.annotate_dataset_chains_with_kinome(
    "Results/InterPro_protein_chains/",
    output_csv="Results/kinase_annotation_all_chains.csv",
)
display(annot_all.head())

figs = lab.plot_annotation_summary(annot_all, species_top_n=15, save_dir="Results")
for key in ("fig_group", "fig_species", "fig_pseudokinase"):
    display(figs[key])


As expected, our dataset includes kinase structures from all different kinase families.

## 2.2 Filtering for activation loop  <a id="22"></a>
Here we exclude all kinase domains that do not have the characteristic conserved residue motifs DFG and APE that delimit the activation loop.

We utilise `copy_motif_filtered_datasets()` to keep chains that contain DFG and APE (excluding pseudokinases), writing protein-only PDBs to `Results/motif_filtered_chains/` and mirroring the same basenames from the small-molecule extraction into `Results/motif_filtered_small_molecules/`.

In [ ]:
valid_pdbs, invalid_pdbs = copy_motif_filtered_datasets(
    source_dir_protein="Results/InterPro_protein_chains/",
    target_dir_protein="Results/motif_filtered_chains/",
    source_dir_small_molecules="Results/InterPro_protein_small_molecules/",
    target_dir_small_molecules="Results/motif_filtered_small_molecules/",
    excluded_basenames_file="Results/excluded_pseudokinase_basenames.txt",
)


Let's check how many motif-filtered protein chains and protein–small-molecule complexes we are left with.


In [ ]:
pdb_directory = 'Results/motif_filtered_chains/'
pdb_directory2 = 'Results/motif_filtered_small_molecules/'
pdb_count = count_pdb_files(pdb_directory)
pdb_count2 = count_pdb_files(pdb_directory2)

print(f"There are {pdb_count} PDB files in the directory '{pdb_directory}'.")
print(f"There are {pdb_count2} PDB files in the directory '{pdb_directory2}'.")


### 2.3.1 Gap-length filter  <a id="231gap"></a>
Crystal structures sometimes have missing residues in the activation loop that cannot be resolved by MODELLER. Chains are excluded here if the DFG–APE loop has more than **4 consecutive missing residues** **or** more than **7 missing residues in total** (input: `Results/motif_filtered_chains/`), before the loop-length filter in §2.3 and MODELLER in Workflow 2. Passing chains are written to `Results/gap_filtered_chains/` and a list of excluded basenames is saved as `Results/gap_filtered_chains/gap_excluded.txt`.


In [ ]:
from workflow.reconstruct import ProteinReconstructor

gap_filter = ProteinReconstructor(
    input_dir="Results/motif_filtered_chains/",
    full_pdb_dir="Results/InterPro_PDBs/",
    output_dir="Results/gap_filtered_chains/",
    max_gap_length=4,
    max_missing_residues=7,
)

gap_results = gap_filter.filter_by_max_gap()


## 2.3 Tukey activation-loop length filter  <a id="23bounds"></a>
We exclude structures whose **activation-loop sequence length** (SEQRES DFG→APE inclusive, counting missing residues) falls outside Tukey IQR bounds with production **`k_factor = 0.8`**:

$$\text{lower} = Q_1 - k \cdot \mathrm{IQR}, \qquad \text{upper} = Q_3 + k \cdot \mathrm{IQR}$$

On each run the loop TSV is rebuilt from full PDBs in `Results/InterPro_PDBs/` for the chains in `Results/gap_filtered_chains/`, then written to `Results/activation_loop_sequences.tsv`. PDBs that pass are written to `Results/Bounds_CAfilter_chains/` (name kept for downstream notebooks).

The previous fixed-window “eyeball” filter (18–32 residues) is retained as a comparison baseline in [§2.4](#24tukey).


In [ ]:
from workflow.ca_stripper import OutlierStripper

INPUT_CHAINS_DIR = "Results/gap_filtered_chains/"
OUTPUT_CHAINS_DIR = "Results/Tukey_CAfilter_chains/"
FULL_PDB_DIR = "Results/InterPro_PDBs/"
WORKFLOW_K = 0.8

length_filter_results = OutlierStripper(k_factor=WORKFLOW_K, verbose=False).filter_chains_by_loop_length(
    input_chains_dir=INPUT_CHAINS_DIR,
    output_chains_dir=OUTPUT_CHAINS_DIR,
    full_pdb_dir=FULL_PDB_DIR,
    # omit length_lower/upper → Tukey via self.k_factor
)

Let's check how many kinase domains we are left with.

In [ ]:
pdb_directory = 'Results/Tukey_CAfilter_chains/'
pdb_count = count_pdb_files(pdb_directory)

print(f"There are {pdb_count} PDB files in the directory '{pdb_directory}'.")

## 2.4 Tukey k-factor scan and eyeball comparison  <a id="24tukey"></a>

Scan Tukey `k` over activation-loop lengths (metrics only — no PDB copying), analyse retention/bounds, then compare production Tukey (`k=0.8`) to the fixed **18–32** eyeball window. Outputs under `Results/tukey_k_scan/`.


### 2.4.0 Setup: paths and k grid <a id="2-4-0-setup-paths-and-k-grid"></a>


In [ ]:
from workflow.ca_stripper import OutlierStripper

WORKFLOW_K = 0.8
STRICT_K = 1.5
EYEBALL_LOWER, EYEBALL_UPPER = 18, 32
K_VALUES = np.round(np.arange(0.2, 2.51, 0.1), 1)

LOOP_TSV = "Results/activation_loop_sequences.tsv"
EXPERIMENT_DIR = "Results/tukey_k_scan"
PLOT_DIR = os.path.join(EXPERIMENT_DIR, "plots")
SCAN_CSV = os.path.join(EXPERIMENT_DIR, "k_factor_scan_summary.csv")
SCAN_PKL = os.path.join(EXPERIMENT_DIR, "k_factor_scan_summary.pkl")

os.makedirs(EXPERIMENT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

print(f"Scan grid: {len(K_VALUES)} values from {K_VALUES[0]} to {K_VALUES[-1]}")
print(f"Workflow default k = {WORKFLOW_K}")
print(f"Eyeball bounds     = [{EYEBALL_LOWER}, {EYEBALL_UPPER}]")
print(f"Experiment dir     = {EXPERIMENT_DIR}")


### 2.4.1 Load loop-length distribution <a id="2-4-1-load-loop-length-distribution"></a>

Read `Results/activation_loop_sequences.tsv` rebuilt by the production filter in §2.3.


In [ ]:
if not os.path.isfile(LOOP_TSV):
    raise FileNotFoundError(
        f"Missing {LOOP_TSV}. Run §2.3 Tukey loop-length filter first."
    )

loop_df = pd.read_csv(LOOP_TSV, sep="\t")
loop_df["loop_length"] = loop_df["loop_sequence"].astype(str).str.len()
lengths = loop_df["loop_length"].values
names = loop_df["pdb_basename"].astype(str).values
n_total = len(lengths)

Q1, Q3 = np.percentile(lengths, [25, 75])
IQR = Q3 - Q1

print(f"Structures : {n_total}")
print(f"Loop length: min={lengths.min()}, max={lengths.max()}, median={np.median(lengths):.1f}")
print(f"Quartiles  : Q1={Q1:.1f}, Q3={Q3:.1f}, IQR={IQR:.1f}")


### 2.4.2 k-factor parameter scan <a id="2-4-2-k-factor-parameter-scan"></a>

For each `k_factor`, apply Tukey outlier detection to the loop-length vector (fast; no PDB I/O). Results are saved to `Results/tukey_k_scan/`.


In [ ]:
scan_out = OutlierStripper.scan_tukey_k_factors(
    lengths,
    names,
    K_VALUES,
    experiment_dir=EXPERIMENT_DIR,
    scan_csv=SCAN_CSV,
    scan_pkl=SCAN_PKL,
    verbose=True,
)
scan_df = scan_out["scan_df"]
scan_rows = scan_out["scan_rows"]

# Attach loop_df into the pickle for re-analysis convenience
with open(SCAN_PKL, "wb") as f:
    pickle.dump(
        {
            "scan_rows": scan_rows,
            "loop_df": loop_df,
            "K_VALUES": K_VALUES,
            "lengths": lengths,
            "names": names,
        },
        f,
    )
print(f"Updated details pickle with loop_df → {SCAN_PKL}")


### 2.4.3 Load pre-computed scan results <a id="2-4-3-load-pre-computed-scan-results"></a>

Re-run the analysis sections below without re-scanning.


In [ ]:
scan_df = pd.read_csv(SCAN_CSV)

with open(SCAN_PKL, "rb") as f:
    loaded = pickle.load(f)

scan_rows = loaded["scan_rows"]
loop_df = loaded["loop_df"]
lengths = loop_df["loop_length"].values
names = loop_df["pdb_basename"].astype(str).values
n_total = len(lengths)
if "K_VALUES" in loaded:
    K_VALUES = loaded["K_VALUES"]

print(f"Loaded {len(scan_df)} k values from {SCAN_CSV}")


### 2.4.4 Analyse scan <a id="2-4-4-analyse-scan"></a>


#### Retention vs k


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(scan_df["k_factor"], scan_df["n_passing"], "o-", label="Passing", color="mediumseagreen")
ax.plot(scan_df["k_factor"], scan_df["n_removed"], "s-", label="Removed", color="indianred")
ax.axvline(WORKFLOW_K, color="gray", linestyle="--", linewidth=1, label=f"Workflow k={WORKFLOW_K}")
ax.set_xlabel("k-factor (IQR multiplier)")
ax.set_ylabel("Structure count")
ax.set_title("Tukey loop-length filter: retention vs k-factor")
ax.legend()
plt.tight_layout()
out = os.path.join(PLOT_DIR, "retention_vs_k.png")
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {out}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(scan_df["k_factor"], scan_df["pct_passing"], "o-", color="steelblue")
ax.axvline(WORKFLOW_K, color="gray", linestyle="--", linewidth=1, label=f"Workflow k={WORKFLOW_K}")
ax.set_xlabel("k-factor")
ax.set_ylabel("Passing (%)")
ax.set_ylim(0, 101)
ax.set_title("Fraction of structures retained")
ax.legend()
plt.tight_layout()
out = os.path.join(PLOT_DIR, "pct_passing_vs_k.png")
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {out}")


#### Tukey bounds vs k


In [ ]:
_q1, _q3 = np.percentile(lengths, [25, 75])

fig, ax = plt.subplots(figsize=(8, 4))
ax.fill_between(
    scan_df["k_factor"], scan_df["lower_bound"], scan_df["upper_bound"],
    alpha=0.25, color="steelblue", label="Accepted range",
)
ax.plot(scan_df["k_factor"], scan_df["lower_bound"], "--", color="red", label="Lower bound")
ax.plot(scan_df["k_factor"], scan_df["upper_bound"], "--", color="red", label="Upper bound")
ax.axhline(_q1, color="black", linestyle=":", linewidth=1, label=f"Q1={_q1:.1f}")
ax.axhline(_q3, color="black", linestyle=":", linewidth=1, label=f"Q3={_q3:.1f}")
ax.axhline(EYEBALL_LOWER, color="darkorange", linestyle="-.", linewidth=1, label=f"Eyeball lower={EYEBALL_LOWER}")
ax.axhline(EYEBALL_UPPER, color="darkorange", linestyle="-.", linewidth=1, label=f"Eyeball upper={EYEBALL_UPPER}")
ax.axvline(WORKFLOW_K, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("k-factor")
ax.set_ylabel("Activation loop length (residues)")
ax.set_title("Tukey acceptance window vs k-factor")
ax.legend(loc="best", fontsize=8)
plt.tight_layout()
out = os.path.join(PLOT_DIR, "bounds_vs_k.png")
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {out}")


#### Removed structures: too short vs too long


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(scan_df["k_factor"], scan_df["n_too_short"], width=0.08, label="Too short", color="coral")
ax.bar(
    scan_df["k_factor"], scan_df["n_too_long"], width=0.08,
    bottom=scan_df["n_too_short"], label="Too long", color="mediumpurple",
)
ax.axvline(WORKFLOW_K, color="gray", linestyle="--", linewidth=1, label=f"Workflow k={WORKFLOW_K}")
ax.set_xlabel("k-factor")
ax.set_ylabel("Removed count")
ax.set_title("Outliers by direction")
ax.legend()
plt.tight_layout()
out = os.path.join(PLOT_DIR, "removed_split_vs_k.png")
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {out}")


#### Loop-length histogram with cutoffs

Overlay Tukey cutoffs for representative k values (including the workflow default) and the eyeball window.


In [ ]:
HIGHLIGHT_K = [0.4, 0.8, 1.0, 1.5, 2.0]
colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(HIGHLIGHT_K)))

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(lengths, bins=50, color="lightgray", edgecolor="white", label=f"All (n={n_total})")

for k, c in zip(HIGHLIGHT_K, colors):
    row = scan_df.loc[np.isclose(scan_df["k_factor"], k)].iloc[0]
    ax.axvline(row["lower_bound"], color=c, linestyle="--", linewidth=1.2)
    ax.axvline(
        row["upper_bound"], color=c, linestyle="--", linewidth=1.2,
        label=f"k={k:g} [{row['lower_bound']:.0f}, {row['upper_bound']:.0f}]",
    )

ax.axvline(EYEBALL_LOWER, color="darkorange", linestyle="-.", linewidth=1.5)
ax.axvline(
    EYEBALL_UPPER, color="darkorange", linestyle="-.", linewidth=1.5,
    label=f"Eyeball [{EYEBALL_LOWER}, {EYEBALL_UPPER}]",
)

ax.set_xlabel("Activation loop length (residues)")
ax.set_ylabel("Count")
ax.set_title("Loop-length distribution with Tukey windows")
ax.legend(fontsize=7, loc="upper right")
plt.tight_layout()
out = os.path.join(PLOT_DIR, "loop_length_histogram_cutoffs.png")
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {out}")


#### Structures removed only at strict k

Compare the set of removed chains at the workflow default `k=0.8` vs classical Tukey `k=1.5`.


In [ ]:
def removed_set_for_k(k):
    for row in scan_rows:
        if float(row["k_factor"]) == float(k):
            return set(row["removed_basenames"])
    raise KeyError(f"k={k} not in scan results")

rem_workflow = removed_set_for_k(WORKFLOW_K)
rem_strict = removed_set_for_k(STRICT_K)

only_at_strict = rem_strict - rem_workflow
only_at_workflow = rem_workflow - rem_strict
both = rem_workflow & rem_strict

print(f"Removed at k={WORKFLOW_K} (workflow): {len(rem_workflow)}")
print(f"Removed at k={STRICT_K} (classical):  {len(rem_strict)}")
print(f"  In both:              {len(both)}")
print(f"  Only at k={STRICT_K}:       {len(only_at_strict)}")
print(f"  Only at k={WORKFLOW_K}:       {len(only_at_workflow)}")

if only_at_strict:
    extra = loop_df[loop_df["pdb_basename"].isin(only_at_strict)][["pdb_basename", "loop_length"]]
    display(extra.sort_values("loop_length").head(20))

summary_k = scan_df[scan_df["k_factor"].isin([WORKFLOW_K, STRICT_K])].copy()
display(summary_k[[
    "k_factor", "lower_bound", "upper_bound",
    "n_passing", "n_removed", "pct_passing",
    "n_too_short", "n_too_long",
]])


### 2.4.5 Eyeball (18–32) vs production Tukey (`k=0.8`) <a id="2-4-5-eyeball-18-32-vs-production-tukey-k-0-8"></a>

Metrics/plots only — does **not** overwrite `Results/Bounds_CAfilter_chains/`.


In [ ]:
cmp = OutlierStripper.compare_fixed_bounds_vs_tukey(
    lengths,
    names,
    length_lower=EYEBALL_LOWER,
    length_upper=EYEBALL_UPPER,
    k_factor=WORKFLOW_K,
    save_dir=EXPERIMENT_DIR,
    verbose=True,
)

display(cmp["summary_df"].round(4))
display(cmp["overlap_df"])

# Overlay both windows on the loop-length histogram
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(lengths, bins=50, color="lightgray", edgecolor="white", label=f"All (n={n_total})")
ax.axvline(cmp["eyeball"]["lower_bound"], color="darkorange", linestyle="-.", linewidth=1.8)
ax.axvline(
    cmp["eyeball"]["upper_bound"], color="darkorange", linestyle="-.", linewidth=1.8,
    label=f"Eyeball [{EYEBALL_LOWER}, {EYEBALL_UPPER}]",
)
ax.axvline(cmp["tukey"]["lower_bound"], color="steelblue", linestyle="--", linewidth=1.8)
ax.axvline(
    cmp["tukey"]["upper_bound"], color="steelblue", linestyle="--", linewidth=1.8,
    label=(
        f"Tukey k={WORKFLOW_K} "
        f"[{cmp['tukey']['lower_bound']:.1f}, {cmp['tukey']['upper_bound']:.1f}]"
    ),
)
ax.set_xlabel("Activation loop length (residues)")
ax.set_ylabel("Count")
ax.set_title("Eyeball vs production Tukey windows")
ax.legend(fontsize=8)
plt.tight_layout()
out = os.path.join(PLOT_DIR, "eyeball_vs_tukey_histogram.png")
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {out}")

# Retention bar chart
fig, ax = plt.subplots(figsize=(6, 4))
methods = ["Eyeball 18–32", f"Tukey k={WORKFLOW_K}"]
passing = [cmp["eyeball"]["n_passing"], cmp["tukey"]["n_passing"]]
removed = [cmp["eyeball"]["n_removed"], cmp["tukey"]["n_removed"]]
x = np.arange(len(methods))
ax.bar(x - 0.15, passing, width=0.3, label="Passing", color="mediumseagreen")
ax.bar(x + 0.15, removed, width=0.3, label="Removed", color="indianred")
ax.set_xticks(x)
ax.set_xticklabels(methods)
ax.set_ylabel("Structure count")
ax.set_title("Eyeball vs Tukey retention")
ax.legend()
plt.tight_layout()
out = os.path.join(PLOT_DIR, "eyeball_vs_tukey_retention.png")
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {out}")
